# LTX-2.3 T2V/I2V 两阶段（蒸馏）— 最简 Colab
仅安装此工作流所需的最小集合：
- **节点**：ComfyUI 本体 + ComfyUI-LTXVideo（唯一必需的节点包）
- **模型**：dev 检查点 (~46GB) + 蒸馏 LoRA (~7GB) + 2× 空间上采样器 (~2GB) + Gemma 3 fp8 文本编码器 (~13GB)，共约 **68GB 磁盘**

> ⚠️ 22B 模型建议使用 **A100 / L4** 运行时（免费 T4 显存不足且磁盘紧张）。
> 依次运行 Cell 1 → 2 → 3 → 4。访问方式为 FRP 内网穿透（在 Colab「密钥」中配置 `VPS_IP` 和 `FRP_TOKEN`），启动后访问 http://cjp.usdream.dpdns.org:8090 。工作流已自动放入界面的 Workflows 列表。

In [ ]:
#@title Cell 1：安装 ComfyUI + 唯一必需节点包
import subprocess, sys
from pathlib import Path

BASE = Path('/content')
COMFY = BASE / 'ComfyUI'

def run(cmd, cwd=None):
    print('>>', ' '.join(map(str, cmd)))
    subprocess.run(list(map(str, cmd)), cwd=cwd, check=True)

# 1) ComfyUI 本体（EmptyLTXVLatentVideo、LTXV 音频/AV 节点、LatentUpscale 等 27 种节点都在本体里）
if not COMFY.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/comfyanonymous/ComfyUI', COMFY])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', COMFY / 'requirements.txt'])

# 2) 唯一必需的节点包：ComfyUI-LTXVideo
#    提供 GemmaAPITextEncode / LTXVImgToVideoConditionOnly / LTXFloatToInt / LTXVTiledVAEDecode
node_dir = COMFY / 'custom_nodes' / 'ComfyUI-LTXVideo'
if not node_dir.exists():
    run(['git', 'clone', '--depth', '1', 'https://github.com/Lightricks/ComfyUI-LTXVideo', node_dir])
run([sys.executable, '-m', 'pip', 'install', '-q', '-r', node_dir / 'requirements.txt'])

# 3) 可选加速：sageattention（失败不影响运行）
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'sageattention'])
SAGE_FLAG = BASE / '.sage_ok'
try:
    import importlib
    importlib.import_module('sageattention')
    SAGE_FLAG.write_text('1')
    print('⚡ sageattention 可用，启动时将启用加速')
except Exception as e:
    SAGE_FLAG.write_text('0')
    print(f'⚠️ sageattention 不可用，回退默认注意力: {e}')

print('\n✅ Cell 1 完成')

In [ ]:
#@title Cell 2：下载 4 个模型文件 + 工作流 JSON（约 68GB，A100 高速盘约 15-25 分钟）
import subprocess, urllib.request
from pathlib import Path

COMFY = Path('/content/ComfyUI')
M = COMFY / 'models'

# 高速下载器
subprocess.run(['apt-get', 'install', '-qq', '-y', 'aria2'], capture_output=True)

def dl(url, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists() and dest.stat().st_size > 0:
        print(f'✔ 已存在，跳过: {dest.name}')
        return
    print(f'⬇ {dest.name}')
    subprocess.run(['aria2c', '-x16', '-s16', '-k1M', '--console-log-level=warn',
                    '--summary-interval=30', '-d', str(dest.parent), '-o', dest.name, url], check=True)

HF_LTX = 'https://huggingface.co/Lightricks/LTX-2.3/resolve/main'

# 1) 主检查点：dev 完整版（工作流三处加载节点都指向它）~46GB
dl(f'{HF_LTX}/ltx-2.3-22b-dev.safetensors', M / 'checkpoints' / 'ltx-2.3-22b-dev.safetensors')

# 2) 蒸馏 LoRA（把 dev 变成少步蒸馏模式）~7GB — 注意 JSON 写死的子目录 ltxv/ltx2/
dl(f'{HF_LTX}/ltx-2.3-22b-distilled-lora-384-1.1.safetensors',
   M / 'loras' / 'ltxv' / 'ltx2' / 'ltx-2.3-22b-distilled-lora-384-1.1.safetensors')

# 3) 第二阶段 2× 空间上采样器 ~2GB
dl(f'{HF_LTX}/ltx-2.3-spatial-upscaler-x2-1.1.safetensors',
   M / 'latent_upscale_models' / 'ltx-2.3-spatial-upscaler-x2-1.1.safetensors')

# 4) Gemma 3 12B 文本编码器（fp8 单文件版）~13GB
#    直接保存为工作流期望的文件名 comfy_gemma_3_12B_it.safetensors，加载时零改动
dl('https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp8_scaled.safetensors',
   M / 'text_encoders' / 'comfy_gemma_3_12B_it.safetensors')

# 5) 官方工作流 JSON → 界面 Workflows 列表
wf_dir = COMFY / 'user' / 'default' / 'workflows'
wf_dir.mkdir(parents=True, exist_ok=True)
wf_url = ('https://raw.githubusercontent.com/Lightricks/ComfyUI-LTXVideo/master/'
          'example_workflows/2.3/LTX-2.3_T2V_I2V_Two_Stage_Distilled.json')
urllib.request.urlretrieve(wf_url, wf_dir / 'LTX-2.3_T2V_I2V_Two_Stage_Distilled.json')
print('✔ 工作流已放入界面 Workflows 列表')

import shutil
free_gb = shutil.disk_usage('/content').free / 1e9
print(f'\n✅ Cell 2 完成（剩余磁盘 {free_gb:.0f}GB）')

In [ ]:
#@title Cell 3：FRP 内网穿透（原方式，可选）
import subprocess
from pathlib import Path

print("=== 🌐 配置 FRP，可选 ===")

try:
    from google.colab import userdata
    VPS_IP = userdata.get("VPS_IP")
    FRP_TOKEN = userdata.get("FRP_TOKEN")
except Exception:
    VPS_IP = None
    FRP_TOKEN = None

if not VPS_IP or not FRP_TOKEN:
    print("⚠️ 没有 VPS_IP / FRP_TOKEN，跳过 FRP")
else:
    FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

    if not FRP_DIR.exists():
        subprocess.run(
            "wget -qO- https://github.com/fatedier/frp/releases/download/v0.56.0/frp_0.56.0_linux_amd64.tar.gz | tar -xz -C /content",
            shell=True,
            check=True
        )

    conf = f'''
serverAddr = "{VPS_IP}"
serverPort = 7000
auth.token = "{FRP_TOKEN}"

[[proxies]]
name = "comfyui_web_colab"
type = "tcp"
localIP = "127.0.0.1"
localPort = 8188
remotePort = 8090
'''
    (FRP_DIR / "frpc.toml").write_text(conf.strip(), encoding="utf-8")
    print("✅ FRP 配置完成")

In [ ]:
#@title Cell 4：启动 ComfyUI（FRP 访问）
import os, subprocess, threading, time
from pathlib import Path

COMFY_DIR = Path("/content/ComfyUI")
FRP_DIR = Path("/content/frp_0.56.0_linux_amd64")

os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] Colab 保活中...")

threading.Thread(target=keep_alive, daemon=True).start()

# 启动 FRP
frpc = FRP_DIR / "frpc"
frpc_conf = FRP_DIR / "frpc.toml"

if frpc.exists() and frpc_conf.exists():
    def start_frpc():
        subprocess.run([str(frpc), "-c", str(frpc_conf)])

    threading.Thread(target=start_frpc, daemon=True).start()
    print("✅ FRP 已启动")
    print("👉 访问: http://cjp.usdream.dpdns.org:8090")
else:
    print("⚠️ 未启用 FRP，仅本地 127.0.0.1:8188")

# ✅ 若 Cell 1 验证 sageattention 可用则启用加速
launch_cmd = ["python", "main.py", "--dont-print-server"]

SAGE_FLAG = Path("/content/.sage_ok")
if SAGE_FLAG.exists() and SAGE_FLAG.read_text().strip() == "1":
    launch_cmd.append("--use-sage-attention")
    print("⚡ 已启用 Sage Attention 加速")
else:
    print("ℹ️ 未启用 Sage Attention，使用默认注意力")

os.chdir(COMFY_DIR)
print("🚀 启动 ComfyUI...（首次加载 22B 模型需要几分钟）")
subprocess.run(launch_cmd)